In [1]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file,'r') as file:
        content = file.read()
    return [chunk for chunk in content.split('\n\n')]

chunks = split_into_chunks('doc.md')
for i, chunk in enumerate(chunks):
    print(i, chunk)

0 Doraemon and the Super Saiyan: The Battle of Space-Time
1 On a typical afternoon, Nobita sat at his desk dazed, staring at a mountain of homework he hadn't even started. Beside him, Doraemon flipped through a manga, sighing occasionally at the boy’s usual unreliability. Suddenly, a blinding light descended from the ceiling, shaking the entire room. From the radiance stepped a golden-haired youth clad in battle armor, exuding an incredible aura. It was Trunks, the Super Saiyan from the future. He spoke with urgent gravity: the future Earth was on the brink of destruction by dark forces, and he had come to seek Doraemon’s help.
2 Doraemon and Nobita were stunned, but they saw a non-negotiable resolve in Trunks' eyes. Trunks explained that the enemy was no ordinary villain, but a being known as "Dark Saiyan"—a clone created by an evil scientist using Vegeta’s modified genes. This foe possessed not only Saiyan combat prowess but also the ability to manipulate distorted time energy. Trunk

In [2]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()


test_embedding = embed_chunk("test content")
print(len(test_embedding))

    

768


In [3]:
embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(embeddings[0])

10
[0.041841890662908554, 0.019945284351706505, 0.05245504900813103, 0.05084593594074249, 0.006512182764708996, -0.07709771394729614, 0.03165764361619949, 0.0018966970965266228, -0.057434264570474625, 0.0014341674977913499, -0.04654354602098465, 0.00575807923451066, 0.03513896092772484, 0.01821649633347988, -0.04621623829007149, 0.017680758610367775, 0.031084774062037468, 0.044214989989995956, -0.07051041722297668, -0.00924696121364832, 0.03614489361643791, 0.022766323760151863, -0.02294551581144333, 0.011269820854067802, 0.04390722140669823, -0.034102294594049454, -0.042949166148900986, -0.015046162530779839, -0.03732955828309059, 0.08060561865568161, 0.026901261880993843, -0.007273086346685886, -0.006519158836454153, 0.08679225295782089, -0.02278498187661171, 0.005697640590369701, 0.06089301407337189, -0.02398032695055008, -0.008101386949419975, -0.0035626008175313473, 0.016270896419882774, 0.015328424982726574, -0.01886163279414177, -0.02181447297334671, -0.022327978163957596, 0.023

In [4]:
import chromadb

chromadb_client = chromadb.EphemeralClient()
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        chromadb_collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

save_embeddings(chunks, embeddings)

In [5]:
def retrieve(query: str, top_k: int) -> List[str]:
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]
    
query = "What are the three tools used by Doraemon?"
retrieved_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

[0] Doraemon and Nobita were stunned, but they saw a non-negotiable resolve in Trunks' eyes. Trunks explained that the enemy was no ordinary villain, but a being known as "Dark Saiyan"—a clone created by an evil scientist using Vegeta’s modified genes. This foe possessed not only Saiyan combat prowess but also the ability to manipulate distorted time energy. Trunks had fought alone for years, meeting only crushing defeat. "Technology is the only weapon missing from my era," he said. "And you happen to possess it."

[1] The tools were: the "Copycat Cape" to grant temporary super-combat abilities, the "Time-Stop Watch" to freeze time for five seconds, and a "Portable Spirit and Time Training Room" that allowed a year of training in just one minute. Nobita was pushed into the room for an intensive session. Though only minutes passed in reality, he endured a full year of grueling cultivation. At first, he wanted to flee, but the memory of Shizuka, his parents, and Doraemon’s trusting gaze 

In [6]:
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, _ in scored_chunks][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

[0] The tools were: the "Copycat Cape" to grant temporary super-combat abilities, the "Time-Stop Watch" to freeze time for five seconds, and a "Portable Spirit and Time Training Room" that allowed a year of training in just one minute. Nobita was pushed into the room for an intensive session. Though only minutes passed in reality, he endured a full year of grueling cultivation. At first, he wanted to flee, but the memory of Shizuka, his parents, and Doraemon’s trusting gaze kept him going. When he emerged, his body and spirit were renewed; his eyes held a newfound maturity and confidence.

[1] The final battle erupted before the Dark Saiyan’s floating fortress. Trunks charged first, unleashing his full power in a head-on clash. Doraemon provided support using the Anywhere Door and various gadgets to create chaos and suppress the enemy's spatial abilities. However, the Dark Saiyan was too overwhelming for Trunks to defeat alone. Just as Trunks was about to be struck down, Nobita donned 

In [7]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    prompt = f"""You are a knowledge assistant. Please generate an accurate answer based on the user’s question and the following passages.

User's question: {query}

Related passages:
{"\n\n".join(chunks)}

Please answer based on the above information and don't talk nonsense."""

    print(f"{prompt}\n\n---\n")

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

answer = generate(query, reranked_chunks)
print(answer)

You are a knowledge assistant. Please generate an accurate answer based on the user’s question and the following passages.

User's question: What are the three tools used by Doraemon?

Related passages:
The tools were: the "Copycat Cape" to grant temporary super-combat abilities, the "Time-Stop Watch" to freeze time for five seconds, and a "Portable Spirit and Time Training Room" that allowed a year of training in just one minute. Nobita was pushed into the room for an intensive session. Though only minutes passed in reality, he endured a full year of grueling cultivation. At first, he wanted to flee, but the memory of Shizuka, his parents, and Doraemon’s trusting gaze kept him going. When he emerged, his body and spirit were renewed; his eyes held a newfound maturity and confidence.

The final battle erupted before the Dark Saiyan’s floating fortress. Trunks charged first, unleashing his full power in a head-on clash. Doraemon provided support using the Anywhere Door and various gadge